In [7]:
pip install -U trl accelerate peft bitsandbytes transformers huggingface_hub  datasets

### Libraries

In [8]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig,PeftConfig
from trl import SFTTrainer
import re
from transformers import (AutoModelForCausalLM,
                      AutoTokenizer,
                      BitsAndBytesConfig,
                      TrainingArguments,
                      pipeline,
                      logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             confusion_matrix)
from sklearn.model_selection import train_test_split

# Load data

In [9]:
filename="./all-data.csv"
df=pd.read_csv(filename,
               names=["sentiment","text"],
               encoding="utf-8", encoding_errors="replace")
df.head()

,sentiment,text
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...


## Data pre-processing

In [10]:
X_train=list()
X_test= list()
for sentiment in ["positive","neutral","negative"]:
  train,test= train_test_split(df[df.sentiment==sentiment],
                               train_size=300,
                               test_size=300,
                               random_state=42)
  X_train.append(train)
  X_test.append(test)
X_train[:3]

[     sentiment                                               text
 228   positive  ( ADP News ) - Feb 12 , 2009 - Finnish IT solu...
 835   positive  Finnish consulting and engineering group Poyry...
 991   positive  These companies will be able to keep their mar...
 2241  positive  The sale will lead to a pretax capital gain of...
 13    positive  Finnish Talentum reports its operating profit ...
 ...        ...                                                ...
 1761  positive  SysOpen Digia Plc , Press release , 7 February...
 234   positive  Neste Oil Corp. has signed long-term procureme...
 2294  positive  Outokumpu of Finland , stainless steel manufac...
 55    positive  Shares of Nokia Corp. rose Thursday after the ...
 745   positive  `` In terms of profitability and earnings 2007...
 
 [300 rows x 2 columns],
      sentiment                                               text
 295    neutral  The firm generated sales of 187 mln eur in 2005 .
 3553   neutral  Coffee will be ser

In [11]:
X_train=pd.concat(X_train).sample(frac=1, random_state=10)
X_test=pd.concat(X_test)
X_train[:1]

,sentiment,text
3683,neutral,Mr Jortikka is president of the base metal div...


In [12]:
eval_idx=[idx for idx in df.index if idx not in list(train.index) + list (test.index)]
eval_idx [:5]

[0, 1, 3, 4, 5]

### converting index to its related data

In [13]:
X_eval=df[df.index.isin(eval_idx)]
X_eval [:5]


,sentiment,text
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...
5,positive,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...


In [14]:
X_eval=(X_eval
        .groupby('sentiment', group_keys=False)
        .apply(lambda x: x.sample(n=50,random_state=10, replace=True)))
X_train=X_train.reset_index(drop=True)


### Transform Data to LLM Format using Dataset Lib

In [15]:
def generate_prompt(data_point):
  return f"""The sentiment of the following phrase: '{data_point["text"]}' is
  \n\n Positive
  \n Negative
  \n Neutral
  \n Cannot be determined
  \n\nSolution: The correct option is {data_point["sentiment"]}""".strip()

def generate_test_prompt(data_point):
  return f"""The sentiment of the following phrase: '{data_point["text"]}' is
  \n\n Positive
  \n Negative
  \n Neutral
  \n Cannot be determined
  \n\nSolution: The correct option is""".strip()



In [16]:
X_train= pd.DataFrame(X_train.apply(generate_prompt,axis=1), columns= ["text"])
X_eval= pd.DataFrame(X_eval.apply(generate_prompt,axis=1), columns=["text"])
X_train.head()

,text
0,The sentiment of the following phrase: 'Mr Jor...
1,The sentiment of the following phrase: 'Both o...
2,The sentiment of the following phrase: 'Finnis...
3,The sentiment of the following phrase: 'Renzo ...
4,The sentiment of the following phrase: '`` We ...


In [17]:
y_true=X_test.sentiment
X_test= pd.DataFrame(X_test.apply(generate_test_prompt, axis=1), columns=["text"])
train_data=Dataset.from_pandas(X_train)
eval_data=Dataset.from_pandas(X_eval)

In [18]:
train_data['text'][1]

"The sentiment of the following phrase: 'Both operating profit and net sales for the 12-month period increased , respectively from EUR21 .5 m and EUR196 .1 m , as compared to 2005 .' is\n  \n\n Positive\n  \n Negative\n  \n Neutral\n  \n Cannot be determined\n  \n\nSolution: The correct option is positive"

### Fine-tuned model evaluation:
- Map lables to numerical format to :
   - calculate accuracy of test data
   - generate accuracy report for each sentiment label_ranking_average_precision_score
   - generate a classification report
   - generate a confusion matrix

In [19]:
def evaluate (y_true,y_pred):
  labels= ['positive','neutral','negative']
  mapping={'positive':2, 'neutral':1, 'none':1, 'negative':0}
  def map_fun(x):
    return mapping.get(x,1)
  y_true= np.vectorize(map_fun)(y_true)
  y_pred= np.vectorize(map_fun)(y_pred)

  #Accuracy:
  accuracy= accuracy_score (y_true=y_true, y_pred=y_pred)
  print (f'Accuracy: {accuracy:.3f}')

  # for generating accuracy report for confusion matrix
  unique_labels= set(y_true)
  #confusion matrix
  for label in unique_labels:
    label_indices= [i for i in range(len(y_true)) if y_true[i]==label]
    label_y_true= [y_true[i] for i in label_indices]
    label_y_pred=[y_pred[i] for i in label_indices]
    accuracy= accuracy_score(label_y_true, label_y_pred)
    print(f'Accuracy for label {label}: {accuracy:.3f}')

  # Generating classification report
  class_report= classification_report(y_true=y_true,y_pred=y_pred)
  print ('\n Classification report:')
  print (class_report)

  #generate confusion matrix:
  conf_matrix= confusion_matrix(y_true=y_true,y_pred=y_pred, labels=[0,1,2])
  print ('\n confusion matrix:')
  print(conf_matrix)

#Loading model

In [20]:
model_name= "microsoft/phi-2"
compute_dtype=getattr(torch,"float16")
# model configurations:
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype
)
# Loading the Model:
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    device_map="auto",
    quantization_config=bnb_config
)
model.config.use_cache=False
model.config.pretraining_tp=1
#loading tokenizer:
tokenizer= AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token=tokenizer.eos_token




/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.7k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


# A function to predict the sentiment:

In [21]:
def predict(X_test,model, tokenizer):
  y_pred=[]
  for i in tqdm(range(len(X_test))):
    prompt= X_test.iloc[i]["text"]
    pipe= pipeline(task="text-generation",
                   model=model,
                   tokenizer=tokenizer,
                   max_new_tokens=3,
                   temperature= 0.0,
                   )
    result=pipe(prompt,pad_token_id=pipe.tokenizer.eos_token_id)
    answer=result[0]['generated_text'].split("The correct option is")[-1].lower()
    if "positive" in answer:
      y_pred.append("positive")
    elif "negative" in answer:
      y_pred.append("negative")
    elif "neutral" in answer:
      y_pred.append("neutral")
    else:
      y_pred.append("none")
  return y_pred


In [22]:
y_pred=predict(X_test, model,tokenizer)

  0%|          | 0/900 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
100%|██████████| 900/900 [02:47<00:00,  5.37it/s]


In [23]:
evaluate(y_true,y_pred)

Accuracy: 0.444
Accuracy for label 0: 0.413
Accuracy for label 1: 0.703
Accuracy for label 2: 0.217

 Classification report:
              precision    recall  f1-score   support

           0       0.75      0.41      0.53       300
           1       0.35      0.70      0.47       300
           2       0.50      0.22      0.30       300

    accuracy                           0.44       900
   macro avg       0.53      0.44      0.43       900
weighted avg       0.53      0.44      0.43       900


 confusion matrix:
[[124 175   1]
 [ 25 211  64]
 [ 17 218  65]]


# Fine tune model

### Get the number of layers and number of heads for phi2

In [24]:
def get_num_layers(model):
  numbers=set()
  for name, _ in model.named_parameters():
    for number in re.findall(r'\d+', name):
      numbers.add(int(number))
  return max(numbers)

def get_last_layer_linears(model):
  names=[]
  num_layers=get_num_layers(model)
  for name, module in model.named_modules():
    if str(num_layers) in name and not "encoder" in name:
      if isinstance(module, tourch.nn.Linear):
        names.append(name)
  return names

###LoRA

In [25]:
peft_config=LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "up_proj",
        "o_proj",
        "k_proj",
        "down_proj",
        "gate_proj",
        "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

### training

In [26]:
training_arguments=TrainingArguments(
    output_dir="logs",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_32bit",
    save_steps=0,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    evaluation_strategy="epoch"
)
trainer=SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
    max_seq_length=512,
)







/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1965: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transfor

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [27]:
trainer.train()


Epoch,Training Loss,Validation Loss
0,1.443600,1.260970


TrainOutput(global_step=112, training_loss=1.6767988801002502, metrics={'train_runtime': 229.0131, 'train_samples_per_second': 3.93, 'train_steps_per_second': 0.489, 'total_flos': 976063722516480.0, 'train_loss': 1.6767988801002502, 'epoch': 0.9955555555555555})

###Save the model

In [28]:
trainer.model.save_pretrained("mic-phi2-sentiment-model-farzad1")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


###Evaluation

In [29]:
y_pred=predict(X_test, model,tokenizer)
evaluate(y_true, y_pred)

  0%|          | 0/900 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
100%|██████████| 900/900 [04:24<00:00,  3.41it/s]

Accuracy: 0.782
Accuracy for label 0: 0.973
Accuracy for label 1: 0.480
Accuracy for label 2: 0.893

 Classification report:
              precision    recall  f1-score   support

           0       0.90      0.97      0.94       300
           1       0.85      0.48      0.61       300
           2       0.66      0.89      0.76       300

    accuracy                           0.78       900
   macro avg       0.80      0.78      0.77       900
weighted avg       0.80      0.78      0.77       900


 confusion matrix:
[[292   5   3]
 [ 19 144 137]
 [ 12  20 268]]


###save CSV

In [30]:
evaluation=pd.DataFrame({'text': X_test["text"],
                         'y_true': y_true,
                         'y_pred': y_pred})
evaluation.to_csv("test_predictions.csv",index=False)

In [31]:
from IPython.display import FileLink

In [35]:
import shutil

model_path="./mic-phi2-sentiment-model-farzad1"
shutil.make_archive(model_path, 'zip', model_path)
FileLink(f"{model_path}.zip")

/content/mic-phi2-sentiment-model-farzad1.zip

In [33]:
tokenizer.save_pretrained(model_path)

('./mic-phi2-sentiment-model-farzad1/tokenizer_config.json',
 './mic-phi2-sentiment-model-farzad1/special_tokens_map.json',
 './mic-phi2-sentiment-model-farzad1/vocab.json',
 './mic-phi2-sentiment-model-farzad1/merges.txt',
 './mic-phi2-sentiment-model-farzad1/added_tokens.json',
 './mic-phi2-sentiment-model-farzad1/tokenizer.json')